In [5]:
import sys
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import os
import gc
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sparse
from scvi.data import synthetic_iid
from scvi.dataloaders import AnnDataLoader
#import diopy
import warnings
warnings.filterwarnings('ignore')
from cell2location import run_colocation
from cell2location.models import Cell2location, RegressionModel
import torch
import cell2location
from matplotlib.lines import Line2D
import umap
from scvi.model import CondSCVI,DestVI
import scvi
import matplotlib as mpl
from anndata import AnnData
import pathlib
import matplotlib as mpl
import skimage
import tangram as tg
sc.logging.print_header()
import torch
torch.set_float32_matmul_precision('high')  # or 'high' for better performance with a potential precision trade-off

scanpy==1.9.2 anndata==0.8.0 umap==0.5.3 numpy==1.22.4 scipy==1.9.1 pandas==1.5.1 scikit-learn==1.2.1 statsmodels==0.13.5 python-igraph==0.10.4 pynndescent==0.5.8


In [ ]:
# T_names list
T_names = ["T906", "T758", "T996", "T756", "T993", "T989"]
for profile in T_names:
    scdir = "/data/snRNA_downsample.h5ad"
    stdir = "/data/"+profile+"_Spatial-id.h5ad"
    key = 'subclass.v4'
    adata_sc= sc.read_h5ad(scdir)
    adata_st= sc.read_h5ad(stdir)
    adata_ref=adata_sc
    adata_ref.X=sparse.csr_matrix(np.round(adata_ref.X.todense()))
    RegressionModel.setup_anndata(adata_ref, labels_key=key)#需要修改
    mod = RegressionModel(adata_ref)
    mod.train(max_epochs=250, batch_size=2000, train_size=1, lr=0.002, use_gpu=True
             )

    adata_ref = mod.export_posterior(
    adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 1500
    #, 'use_gpu': False
    }
    )
    if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
        inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                        for i in adata_ref.uns['mod']['factor_names']]].copy()
    else:
        inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                        for i in adata_ref.uns['mod']['factor_names']]].copy()


    inf_aver.columns = adata_ref.uns['mod']['factor_names']

    adata_vis=adata_st
    intersect = np.intersect1d(adata_vis.var_names, inf_aver.index)
    adata_vis = adata_vis[:, intersect].copy()
    inf_aver = inf_aver.loc[intersect, :].copy()
    Cell2location.setup_anndata(adata=adata_vis)
    mod = cell2location.models.Cell2location(
        adata_vis, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
        N_cells_per_location=30,#改
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection (using default here):
        detection_alpha=200
    )
    mod.train(max_epochs=250,#00,
          # train using full data (batch_size=None)
          batch_size=5000,#改
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1,
          use_gpu=True
          )
    adata_vis = mod.export_posterior(
    adata_vis, sample_kwargs={'num_samples':500, 'batch_size': mod.adata.n_obs#, 'use_gpu': False
        }
    )
    name="/data/work/Layer division/01_Result/Cell2_"+profile+".cellbin_pret"+".csv"#修改
adata_vis.obsm["means_cell_abundance_w_sf"].to_csv(name)
